In [0]:
import datetime
import os
import sys
import importlib
import importlib
import utils.silver_custom_utils

from typing import List
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
from utils.silver_custom_utils import Transformation
from utils.logger import logger
from utils.silver_custom_utils import Transformation


### CUSTOMERS

In [0]:
df_cust = spark.read.table("ride_analytics.bronze.customers")

In [0]:
customer_validation_rules = [
    ("customer_id IS NOT NULL", "customer_id IS NULL"),
    (
        "email RLIKE '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'",
        "invalid Email format"
    ),
    ("phone_number IS NOT NULL", "phone_number IS NULL")
]

In [0]:
cust_obj = Transformation()

In [0]:
valid_cust_df, rejected_cust_df = (
    cust_obj.separate_valid_rejected(
        df_cust,
        customer_validation_rules
    )
)

2026-09-10 19:29:47,130 - INFO - Starting data validation with 3 validation rules
2026-09-10 19:29:48,232 - INFO - Validation completed. Valid records: 993, Rejected records: 7


In [0]:
if rejected_cust_df.count() > 0:

    logger.info("Rejected Customer DataFrame is not empty")

    quarantine_table = "ride_analytics.silver.customers_quarantine"

    if not spark.catalog.tableExists(quarantine_table):

        logger.info(
            "Quarantine table does not exist. Creating table: %s",
            quarantine_table
        )

        rejected_cust_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(quarantine_table)

        logger.info(
            "Quarantine table created successfully: %s",
            quarantine_table
        )

    else:

        logger.info(
            "Quarantine table exists. Appending rejected records: %s",
            quarantine_table
        )

        rejected_cust_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(quarantine_table)

        logger.info(
            "Rejected records appended successfully"
        )

2026-09-10 19:29:49,055 - INFO - Rejected Customer DataFrame is not empty
2026-09-10 19:29:49,239 - INFO - Quarantine table does not exist. Creating table: ride_analytics.silver.customers_quarantine
2026-09-10 19:29:52,307 - INFO - Quarantine table created successfully: ride_analytics.silver.customers_quarantine


In [0]:
df_cust = valid_cust_df # for now doing it

In [0]:
# Extracting the domain name from email and adding it in a new column

df_cust = df_cust.withColumn("domain",split(col('email'),'@')[1])

In [0]:
# Cleaning the phonenumber column
df_cust = df_cust.withColumn(
    "phone_number",
    expr(
        "right(regexp_replace(phone_number, '[^0-9]', ''), 10)"
    )
)

In [0]:
# Trim unnecessary spaces from customer string columns

df_cust = (
    df_cust
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("email", trim(col("email")))
    .withColumn("city", trim(col("city")))
)


In [0]:
# Standardize email to lowercase

df_cust = df_cust.withColumn(
    "email",
    lower(col("email"))
)


In [0]:
# Standardize city name

df_cust = df_cust.withColumn(
    "city",
    initcap(col("city"))
)

In [0]:
# Concating the first and last name in a new column: Full Name and dropping the first and last name

df_cust = df_cust.withColumn("full_name",concat_ws(" ",col('first_name'),col('last_name')))

df_cust = df_cust.drop('first_name',"last_name")


In [0]:
# Create signup_year from signup_date

df_cust = df_cust.withColumn(
    "signup_year",
    year(col("signup_date"))
)


In [0]:
# Create signup_month from signup_date

df_cust = df_cust.withColumn(
    "signup_month",
    month(col("signup_date"))
)

In [0]:
# Calculate customer tenure in years

df_cust = df_cust.withColumn(
    "customer_tenure_years",
    floor(
        months_between(
            current_date(),
            col("signup_date")
        ) / 12
    )
)

In [0]:
# Creating an object and calling the generic functions dedup & process_timestamp


df_cust = cust_obj.dedup(df_cust,['customer_id'],'last_updated_timestamp')
df_cust = cust_obj.process_timestamp(df_cust)

2026-09-10 19:29:56,430 - INFO - Starting deduplication using columns: ['customer_id']
2026-09-10 19:29:57,336 - INFO - Deduplication completed. Input records: 993, Output records: 978, Duplicates removed: 15
2026-09-10 19:29:57,337 - INFO - Adding process timestamp to DataFrame
2026-09-10 19:29:57,338 - INFO - Process timestamp added successfully


In [0]:
# Check whether Silver table already exists
silver_table = "ride_analytics.silver.customers"

logger.info("Checking Silver table: %s", silver_table)

# Capture processing start time
start_time = datetime.now()


if not spark.catalog.tableExists(silver_table):

    # First run: create the Silver table
    logger.info("Starting FULL LOAD for %s",silver_table)

    df_cust.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(silver_table)

    logger.info(
        "Silver table created successfully: %s",
        silver_table
    )

else:

    # Subsequent runs: perform incremental upsert
    logger.info("Starting INCREMENTAL UPSERT for %s",silver_table)

    result = cust_obj.upsert(
        df_cust,
        ["customer_id"],
        silver_table,
        "last_updated_timestamp"
    )



    logger.info(
        "Incremental upsert completed. Merge condition: %s",
        result
    )

# Calculate total processing time
end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

logger.info(
    "Silver Customers processing completed in %.2f seconds",
    duration
)

2026-09-10 19:29:57,726 - INFO - Checking Silver table: ride_analytics.silver.customers
2026-09-10 19:29:57,888 - INFO - Starting FULL LOAD for ride_analytics.silver.customers
2026-09-10 19:30:01,029 - INFO - Silver table created successfully: ride_analytics.silver.customers
2026-09-10 19:30:01,031 - INFO - Silver Customers processing completed in 3.30 seconds


### DRIVERS

In [0]:
df_drivers = spark.read.table("ride_analytics.bronze.drivers")


In [0]:
drivers_obj = Transformation()

In [0]:
driver_validation_rules = [
    ("driver_id IS NOT NULL", "driver_id IS NULL"),
    ("vehicle_id IS NOT NULL", "vehicle_id IS NULL"),
    ("driver_rating IS NOT NULL", "driver_rating IS NULL"),
    (
        "driver_rating BETWEEN 0 AND 5",
        "driver_rating must be between 0 and 5"
    ),
    ("phone_number IS NOT NULL", "phone_number IS NULL")
]

In [0]:
valid_drivers_df, rejected_drivers_df = (
    drivers_obj.separate_valid_rejected(
        df_drivers,
        driver_validation_rules
    )
)

2026-09-10 19:30:02,106 - INFO - Starting data validation with 5 validation rules
2026-09-10 19:30:03,341 - INFO - Validation completed. Valid records: 235, Rejected records: 15


In [0]:
if rejected_drivers_df.count() > 0:

    logger.info("Rejected Drivers DataFrame is not empty")

    quarantine_table = "ride_analytics.silver.drivers_quarantine"

    if not spark.catalog.tableExists(quarantine_table):

        logger.info(
            "Drivers Quarantine table does not exist. Creating table: %s",
            quarantine_table
        )

        rejected_drivers_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(quarantine_table)

        logger.info(
            "Drivers Quarantine table created successfully: %s",
            quarantine_table
        )

    else:

        logger.info(
            "Drivers Quarantine table exists. Appending rejected records"
        )

        rejected_drivers_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(quarantine_table)

        logger.info(
            "Rejected driver records appended successfully"
        )

2026-09-10 19:30:04,198 - INFO - Rejected Drivers DataFrame is not empty
2026-09-10 19:30:04,367 - INFO - Drivers Quarantine table does not exist. Creating table: ride_analytics.silver.drivers_quarantine
2026-09-10 19:30:07,411 - INFO - Drivers Quarantine table created successfully: ride_analytics.silver.drivers_quarantine


In [0]:
df_drivers = valid_drivers_df

In [0]:
# Remove unnecessary spaces from string columns

df_drivers = (
    df_drivers
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("city", trim(col("city")))
)

In [0]:
# Standardize city names

df_drivers = df_drivers.withColumn(
    "city",
    initcap(col("city"))
)

In [0]:
#Cleaning the phone number

df_drivers = df_drivers.withColumn(
    "phone_number",
    expr(
        "right(regexp_replace(phone_number, '[^0-9]', ''), 10)"
    )
)

In [0]:
#Concating the first and lasrt name in a new column : Full Name and dropping the first and last name

df_drivers = df_drivers.withColumn("full_name",concat_ws(" ",col("first_name"),col("last_name")))
df_drivers = df_drivers.drop("first_name","last_name") 


In [0]:
# Categorize drivers based on their rating

df_drivers = df_drivers.withColumn(
    "rating_category",
    when(col("driver_rating") >= 4.5, "Excellent")
    .when(col("driver_rating") >= 4.0, "Good")
    .when(col("driver_rating") >= 3.0, "Average")
    .otherwise("Poor")
)

In [0]:
# Deduplicate records and add processing timestamp

df_drivers = drivers_obj.dedup(df_drivers,["driver_id"],"last_updated_timestamp")

df_drivers = drivers_obj.process_timestamp(df_drivers)




2026-09-10 19:30:09,729 - INFO - Starting deduplication using columns: ['driver_id']
2026-09-10 19:30:10,634 - INFO - Deduplication completed. Input records: 235, Output records: 225, Duplicates removed: 10
2026-09-10 19:30:10,635 - INFO - Adding process timestamp to DataFrame
2026-09-10 19:30:10,635 - INFO - Process timestamp added successfully


In [0]:
# Check whether Silver Drivers table already exists
silver_table = "ride_analytics.silver.drivers"

logger.info("Checking Silver table: %s", silver_table)


# Capture start time
start_time = datetime.now()

if not spark.catalog.tableExists(silver_table):

    # First run: create the Silver table
    logger.info("Starting FULL LOAD for Silver Drivers")

    df_drivers.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(silver_table)

    logger.info(
        "Silver table created successfully: %s",
        silver_table
    )

else:

    # Subsequent runs: perform incremental upsert
    logger.info("Starting INCREMENTAL UPSERT for Silver Drivers")


    # Execute upsert
    result = drivers_obj.upsert(
        df=df_drivers,
        key_cols=["driver_Id"],
        table_name=silver_table,
        cdc="last_updated_timestamp"
    )

    logger.info(
        "Incremental upsert completed. Merge condition: %s",
        result
    )

    # Capture finish time and calculate duration
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()

    logger.info(
        "Silver Drivers upsert completed in %.2f seconds",
        duration
    )

2026-09-10 19:30:11,036 - INFO - Checking Silver table: ride_analytics.silver.drivers
2026-09-10 19:30:11,197 - INFO - Starting FULL LOAD for Silver Drivers
2026-09-10 19:30:14,200 - INFO - Silver table created successfully: ride_analytics.silver.drivers


### LOCATIONS


In [0]:
df_loc = spark.read.table("ride_analytics.bronze.locations")

In [0]:
loc_obj = Transformation()

In [0]:
location_validation_rules = [
    ("location_id IS NOT NULL", "location_id IS NULL"),
    ("city IS NOT NULL", "city IS NULL"),
    (
        "latitude BETWEEN -90 AND 90",
        "latitude must be between -90 and 90"
    ),
    (
        "longitude BETWEEN -180 AND 180",
        "longitude must be between -180 and 180"
    )
]

In [0]:
valid_loc_df, rejected_loc_df = (
    loc_obj.separate_valid_rejected(
        df_loc,
        location_validation_rules
    )
)

2026-09-10 19:30:15,770 - INFO - Starting data validation with 4 validation rules
2026-09-10 19:30:16,929 - INFO - Validation completed. Valid records: 237, Rejected records: 13


In [0]:
if rejected_loc_df.count() > 0:

    logger.info("Rejected Locations DataFrame is not empty")

    quarantine_table = "ride_analytics.silver.locations_quarantine"

    if not spark.catalog.tableExists(quarantine_table):

        logger.info(
            "Locations Quarantine table does not exist. Creating table: %s",
            quarantine_table
        )

        rejected_loc_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(quarantine_table)

        logger.info(
            "Locations Quarantine table created successfully: %s",
            quarantine_table
        )

    else:

        logger.info(
            "Locations Quarantine table exists. Appending rejected records"
        )

        rejected_loc_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(quarantine_table)

        logger.info(
            "Rejected location records appended successfully"
        )

2026-09-10 19:30:17,745 - INFO - Rejected Locations DataFrame is not empty
2026-09-10 19:30:17,945 - INFO - Locations Quarantine table does not exist. Creating table: ride_analytics.silver.locations_quarantine
2026-09-10 19:30:21,072 - INFO - Locations Quarantine table created successfully: ride_analytics.silver.locations_quarantine


In [0]:
df_loc = valid_loc_df

In [0]:

# Remove unnecessary spaces from location text columns
df_loc = (
    df_loc
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    .withColumn("country", trim(col("country")))
)

In [0]:
# Standardize text format

df_loc = (
    df_loc
    .withColumn("city", initcap(col("city")))
    .withColumn("state", initcap(col("state")))
    .withColumn("country", initcap(col("country")))
)

In [0]:
# Create a readable location name

df_loc = df_loc.withColumn(
    "location_name",
    concat_ws(
        ", ",
        col("city"),
        col("state"),
        col("country")
    )
)

In [0]:

df_loc = loc_obj.dedup(df_loc,["location_id"],"last_updated_timestamp")
df_loc = loc_obj.process_timestamp(df_loc)



2026-09-10 19:30:22,948 - INFO - Starting deduplication using columns: ['location_id']
2026-09-10 19:30:23,865 - INFO - Deduplication completed. Input records: 237, Output records: 232, Duplicates removed: 5
2026-09-10 19:30:23,866 - INFO - Adding process timestamp to DataFrame
2026-09-10 19:30:23,867 - INFO - Process timestamp added successfully


In [0]:
silver_table = "ride_analytics.silver.locations"

# Check whether Silver table already exists
logger.info("Checking Silver table: %s", silver_table)

# Capture processing start time
start_time = datetime.now()

if not spark.catalog.tableExists(silver_table):

    # First run: create the Silver table
    logger.info("Starting FULL LOAD for Silver Locations")

    df_loc.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(silver_table)

    logger.info(
        "Silver table created successfully: %s",
        silver_table
    )

else:

    # Subsequent runs: perform incremental upsert
    logger.info("Starting INCREMENTAL UPSERT for Silver Locations")

    result = loc_obj.upsert(
        df_loc,
        ["location_id"],
        silver_table,
        "last_updated_timestamp"
    )

    logger.info(
        "Incremental upsert completed. Merge condition: %s",
        result
    )

# Calculate total processing time
end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

logger.info(
    "Silver Locations processing completed in %.2f seconds",
    duration
)

2026-09-10 19:30:24,253 - INFO - Checking Silver table: ride_analytics.silver.locations
2026-09-10 19:30:24,445 - INFO - Starting FULL LOAD for Silver Locations
2026-09-10 19:30:27,524 - INFO - Silver table created successfully: ride_analytics.silver.locations
2026-09-10 19:30:27,525 - INFO - Silver Locations processing completed in 3.27 seconds


### PAYMENTS

In [0]:
df_pay = spark.read.table("ride_analytics.bronze.payments")

In [0]:
payments_obj = Transformation()

In [0]:
payment_validation_rules = [
    ("payment_id IS NOT NULL", "payment_id IS NULL"),
    ("trip_id IS NOT NULL", "trip_id IS NULL"),
    ("customer_id IS NOT NULL", "customer_id IS NULL"),
    ("amount IS NOT NULL", "amount IS NULL"),
    (
        "amount >= 0",
        "payment amount cannot be negative"
    ),
    ("payment_method IS NOT NULL", "payment_method IS NULL")
]

In [0]:
valid_pay_df, rejected_pay_df = (
    payments_obj.separate_valid_rejected(
        df_pay,
        payment_validation_rules
    )
)

2026-09-10 19:30:28,599 - INFO - Starting data validation with 6 validation rules
2026-09-10 19:30:29,740 - INFO - Validation completed. Valid records: 4988, Rejected records: 12


In [0]:
if rejected_pay_df.count() > 0:

    logger.info("Rejected Payments DataFrame is not empty")

    quarantine_table = "ride_analytics.silver.payments_quarantine"

    if not spark.catalog.tableExists(quarantine_table):

        logger.info(
            "Payments Quarantine table does not exist. Creating table: %s",
            quarantine_table
        )

        rejected_pay_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(quarantine_table)

        logger.info(
            "Payments Quarantine table created successfully: %s",
            quarantine_table
        )

    else:

        logger.info(
            "Payments Quarantine table exists. Appending rejected records"
        )

        rejected_pay_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(quarantine_table)

        logger.info(
            "Rejected payment records appended successfully"
        )

2026-09-10 19:30:30,469 - INFO - Rejected Payments DataFrame is not empty
2026-09-10 19:30:30,659 - INFO - Payments Quarantine table does not exist. Creating table: ride_analytics.silver.payments_quarantine
2026-09-10 19:30:33,569 - INFO - Payments Quarantine table created successfully: ride_analytics.silver.payments_quarantine


In [0]:
df_pay = valid_pay_df


In [0]:
# Remove spaces and standardize payment method and status
# If the value is Upi, standardize it to uppercase UPI


df_pay = (
    df_pay
    .withColumn(
        "payment_method",
        when(
            upper(trim(col("payment_method"))) == "UPI",
            "UPI"
        ).otherwise(
            initcap(trim(col("payment_method")))
        )
    )
    .withColumn(
        "payment_status",
        initcap(trim(col("payment_status")))
    )
)

In [0]:
# Classify payment based on payment method and payment status

df_pay = df_pay.withColumn(
    "online_payment_status",
    when(
        (col("payment_method").isin("Card", "Wallet","UPI")) &
        (col("payment_status") == "Success"),
        "Online-Success"
    )
    .when(
        (col("payment_method").isin("Card", "Wallet","UPI")) &
        (col("payment_status") == "Pending"),
        "Online-Pending"
    )
    .when(
        (col("payment_method").isin("Card", "Wallet","UPI")) &
        (col("payment_status") == "Failed"),
        "Online-Failed"
    )
    .when(
        col("payment_method") == "Cash",
        "Offline"
    )
    .otherwise("Other")
)

In [0]:
df_pay.display()

payment_id,trip_id,customer_id,payment_method,payment_status,amount,transaction_time,last_updated_timestamp,ingestion_timestamp,online_payment_status
1.0,1.0,1.0,Wallet,Success,247.71,2025-08-08T10:51:00.000Z,2025-08-18T19:00:00.000Z,2026-09-10T19:29:12.227Z,Online-Success
2.0,2.0,2.0,Cash,Success,637.68,2025-08-21T09:21:00.000Z,2025-07-02T14:00:00.000Z,2026-09-10T19:29:12.227Z,Offline
3.0,3.0,3.0,Cash,Success,453.75,2025-09-09T14:30:00.000Z,2025-07-25T08:00:00.000Z,2026-09-10T19:29:12.227Z,Offline
4.0,4.0,4.0,Wallet,Success,460.39,2025-07-27T15:22:00.000Z,2025-07-13T19:00:00.000Z,2026-09-10T19:29:12.227Z,Online-Success
5.0,5.0,5.0,Card,Pending,691.06,2025-08-04T21:31:00.000Z,2025-07-03T03:00:00.000Z,2026-09-10T19:29:12.227Z,Online-Pending
6.0,6.0,6.0,Card,Success,410.17,2025-08-30T01:04:00.000Z,2025-08-06T04:00:00.000Z,2026-09-10T19:29:12.227Z,Online-Success
7.0,7.0,7.0,Wallet,Success,215.38,2025-08-23T17:57:00.000Z,2025-08-02T16:00:00.000Z,2026-09-10T19:29:12.227Z,Online-Success
8.0,8.0,8.0,Cash,Success,357.05,2025-08-13T08:02:00.000Z,2025-07-04T23:00:00.000Z,2026-09-10T19:29:12.227Z,Offline
9.0,9.0,9.0,UPI,Pending,264.12,2025-09-08T03:23:00.000Z,2025-08-28T23:00:00.000Z,2026-09-10T19:29:12.227Z,Online-Pending
10.0,10.0,10.0,Cash,Success,360.68,2025-09-22T11:03:00.000Z,2025-09-08T11:00:00.000Z,2026-09-10T19:29:12.227Z,Offline


In [0]:
# Categorize payments based on transaction amount

df_pay = df_pay.withColumn(
    "payment_amount_category",
    when(col("amount") < 500, "Low")
    .when(col("amount") < 2000, "Medium")
    .otherwise("High")
)

In [0]:
# Extract transaction date from transaction timestamp
df_pay = df_pay.withColumn(
    "transaction_date",
    to_date(col("transaction_time"))
)

In [0]:
# Deduplicate records and add processing timestamp
df_pay = payments_obj.dedup(df_pay,["payment_id"],"last_updated_timestamp")
df_pay = payments_obj.process_timestamp(df_pay)

2026-09-10 19:30:37,122 - INFO - Starting deduplication using columns: ['payment_id']
2026-09-10 19:30:38,061 - INFO - Deduplication completed. Input records: 4988, Output records: 4983, Duplicates removed: 5
2026-09-10 19:30:38,063 - INFO - Adding process timestamp to DataFrame
2026-09-10 19:30:38,067 - INFO - Process timestamp added successfully


In [0]:
silver_table = "ride_analytics.silver.payments"

# Check whether Silver table already exists
logger.info("Checking Silver table: %s", silver_table)

# Capture processing start time
start_time = datetime.now()

if not spark.catalog.tableExists(silver_table):

    # First run: create the Silver table
    logger.info("Starting FULL LOAD for Silver Payments")

    df_pay.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(silver_table)

    logger.info(
        "Silver table created successfully: %s",
        silver_table
    )

else:

    # Subsequent runs: perform incremental upsert
    logger.info("Starting INCREMENTAL UPSERT for Silver Payments")

    result = payments_obj.upsert(
        df_pay,
        ["payment_id"],
        silver_table,
        "last_updated_timestamp"
    )

    logger.info(
        "Incremental upsert completed. Merge condition: %s",
        result
    )

# Calculate total processing time
end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

logger.info(
    "Silver Payments processing completed in %.2f seconds",
    duration
)

2026-09-10 19:30:38,415 - INFO - Checking Silver table: ride_analytics.silver.payments
2026-09-10 19:30:38,630 - INFO - Starting FULL LOAD for Silver Payments
2026-09-10 19:30:42,469 - INFO - Silver table created successfully: ride_analytics.silver.payments
2026-09-10 19:30:42,469 - INFO - Silver Payments processing completed in 4.05 seconds


### VEHICLES

In [0]:
df_veh = spark.read.table("ride_analytics.bronze.vehicles")

In [0]:
veh_obj = Transformation()

In [0]:
vehicle_validation_rules = [
    ("vehicle_id IS NOT NULL", "vehicle_id IS NULL"),
    ("license_plate IS NOT NULL", "license_plate IS NULL"),
    ("model IS NOT NULL", "model IS NULL"),
    ("make IS NOT NULL", "make IS NULL"),
    (
        "year >= 2000",
        "vehicle year must be >= 2000"
    )
]

In [0]:
valid_veh_df, rejected_veh_df = (
    veh_obj.separate_valid_rejected(
        df_veh,
        vehicle_validation_rules
    )
)

2026-09-10 19:30:43,605 - INFO - Starting data validation with 5 validation rules
2026-09-10 19:30:44,640 - INFO - Validation completed. Valid records: 233, Rejected records: 17


In [0]:
if rejected_veh_df.count() > 0:

    logger.info("Rejected Vehicle DataFrame is not empty")

    quarantine_table = "ride_analytics.silver.vehicles_quarantine"

    if not spark.catalog.tableExists(quarantine_table):

        logger.info(
            "Vehicle Quarantine table does not exist. Creating table: %s",
            quarantine_table
        )

        rejected_veh_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(quarantine_table)

        logger.info(
            "Vehicle Quarantine table created successfully: %s",
            quarantine_table
        )

    else:

        logger.info(
            "Vehicle Quarantine table exists. Appending rejected records"
        )

        rejected_veh_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(quarantine_table)

        logger.info(
            "Rejected vehicle records appended successfully"
        )

2026-09-10 19:30:45,389 - INFO - Rejected Vehicle DataFrame is not empty
2026-09-10 19:30:45,577 - INFO - Vehicle Quarantine table does not exist. Creating table: ride_analytics.silver.vehicle_quarantine
2026-09-10 19:30:48,464 - INFO - Vehicle Quarantine table created successfully: ride_analytics.silver.vehicle_quarantine


In [0]:
df_veh = valid_veh_df

In [0]:
# Remove unnecessary spaces from vehicle text columns

df_veh = (
    df_veh
    .withColumn("license_plate", trim(col("license_plate")))
    .withColumn("model", trim(col("model")))
    .withColumn("make", trim(col("make")))
    .withColumn("vehicle_type", trim(col("vehicle_type")))
)

In [0]:
# Standardize vehicle make and type

df_veh = (
    df_veh
    .withColumn("make", initcap(col("make")))
    .withColumn("vehicle_type", initcap(col("vehicle_type")))
)

In [0]:
# Standardize license plate to uppercase

df_veh = df_veh.withColumn(
    "license_plate",
    upper(col("license_plate"))
)

In [0]:
# Calculate current vehicle age

df_veh = df_veh.withColumn(
    "vehicle_age",
    year(current_date()) - col("year")
)

In [0]:
# Categorize vehicles based on their age

df_veh = df_veh.withColumn(
    "vehicle_age_category",
    when(col("vehicle_age") <= 3, "New")
    .when(col("vehicle_age") <= 7, "Mid-Age")
    .otherwise("Old")
)

In [0]:

# Deduplicate records and add processing timestamp

df_veh = veh_obj.dedup(df_veh,["vehicle_id"],"last_updated_timestamp")
df_veh = veh_obj.process_timestamp(df_veh)


2026-09-10 19:30:51,318 - INFO - Starting deduplication using columns: ['vehicle_id']
2026-09-10 19:30:52,172 - INFO - Deduplication completed. Input records: 233, Output records: 228, Duplicates removed: 5
2026-09-10 19:30:52,174 - INFO - Adding process timestamp to DataFrame
2026-09-10 19:30:52,175 - INFO - Process timestamp added successfully


In [0]:
silver_table = "ride_analytics.silver.vehicles"

# Check whether Silver table already exists
logger.info("Checking Silver table: %s", silver_table)

# Capture processing start time
start_time = datetime.now()

if not spark.catalog.tableExists(silver_table):

    # First run: create the Silver table
    logger.info("Starting FULL LOAD for Silver Vehicles")

    df_veh.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(silver_table)

    logger.info(
        "Silver table created successfully: %s",
        silver_table
    )

else:

    # Subsequent runs: perform incremental upsert
    logger.info("Starting INCREMENTAL UPSERT for Silver Vehicles")

    result = veh_obj.upsert(
        df_veh,
        ["vehicle_id"],
        silver_table,
        "last_updated_timestamp"
    )

    logger.info(
        "Incremental upsert completed. Merge condition: %s",
        result
    )

# Calculate total processing time
end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

logger.info(
    "Silver Vehicles processing completed in %.2f seconds",
    duration
)

2026-09-10 19:30:52,525 - INFO - Checking Silver table: ride_analytics.silver.vehicles
2026-09-10 19:30:52,686 - INFO - Starting FULL LOAD for Silver Vehicles
2026-09-10 19:30:55,803 - INFO - Silver table created successfully: ride_analytics.silver.vehicles
2026-09-10 19:30:55,804 - INFO - Silver Vehicles processing completed in 3.28 seconds


### TRIPS

In [0]:
df_trips = spark.read.table("ride_analytics.bronze.trips")

In [0]:
trips_obj = Transformation()

In [0]:
trip_validation_rules = [
    ("trip_id IS NOT NULL", "trip_id IS NULL"),
    ("customer_id IS NOT NULL", "customer_id IS NULL"),
    ("driver_id IS NOT NULL", "driver_id IS NULL"),
    ("vehicle_id IS NOT NULL", "vehicle_id IS NULL"),
    ("distance_km IS NOT NULL", "distance_km IS NULL"),
    (
        "distance_km >= 0",
        "distance cannot be negative"
    ),
    ("fare_amount IS NOT NULL", "fare_amount IS NULL"),
    (
        "fare_amount >= 0",
        "fare amount cannot be negative"
    ),
    ("trip_status IS NOT NULL", "trip_status IS NULL")
]

In [0]:
valid_trips_df, rejected_trips_df = (
    trips_obj.separate_valid_rejected(
        df_trips,
        trip_validation_rules
    )
)

2026-09-10 19:30:57,032 - INFO - Starting data validation with 9 validation rules
2026-09-10 19:30:58,189 - INFO - Validation completed. Valid records: 4979, Rejected records: 21


In [0]:
if rejected_trips_df.count() > 0:

    logger.info("Rejected Trips DataFrame is not empty")

    quarantine_table = "ride_analytics.silver.trips_quarantine"

    if not spark.catalog.tableExists(quarantine_table):

        logger.info(
            "Trips Quarantine table does not exist. Creating table: %s",
            quarantine_table
        )

        rejected_trips_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(quarantine_table)

        logger.info(
            "Trips Quarantine table created successfully: %s",
            quarantine_table
        )

    else:

        logger.info(
            "Trips Quarantine table exists. Appending rejected records"
        )

        rejected_trips_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(quarantine_table)

        logger.info(
            "Rejected trip records appended successfully"
        )

2026-09-10 19:30:58,924 - INFO - Rejected Trips DataFrame is not empty
2026-09-10 19:30:59,084 - INFO - Trips Quarantine table does not exist. Creating table: ride_analytics.silver.trips_quarantine
2026-09-10 19:31:01,983 - INFO - Trips Quarantine table created successfully: ride_analytics.silver.trips_quarantine


In [0]:
df_trips = valid_trips_df

In [0]:
# Remove unnecessary spaces from string columns

df_trips = (
    df_trips
    .withColumn("start_location", trim(col("start_location")))
    .withColumn("end_location", trim(col("end_location")))
    .withColumn("payment_method", trim(col("payment_method")))
    .withColumn("trip_status", trim(col("trip_status")))
)

In [0]:
# Standardize location names

df_trips = (
    df_trips
    .withColumn("start_location", initcap(col("start_location")))
    .withColumn("end_location", initcap(col("end_location")))
)

In [0]:
# Standardize payment method and trip status

df_trips = (
    df_trips
    .withColumn("payment_method", initcap(col("payment_method")))
    .withColumn("trip_status", initcap(col("trip_status")))
)

In [0]:
# Calculate trip duration in minutes

df_trips = df_trips.withColumn(
    "trip_duration_minutes",
    (
        unix_timestamp(col("trip_end_time"))
        - unix_timestamp(col("trip_start_time"))
    ) / 60
)

In [0]:
# Calculate fare per kilometer and round to 2 decimal places

df_trips = df_trips.withColumn(
    "fare_per_km",
    round(
        when(
            col("distance_km") > 0,
            col("fare_amount") / col("distance_km")
        ),
        2
    )
)

In [0]:
# Categorize trips based on distance

df_trips = df_trips.withColumn(
    "distance_category",
    when(col("distance_km") < 5, "Short")
    .when(col("distance_km") < 15, "Medium")
    .otherwise("Long")
)

In [0]:
# Categorize trips based on fare amount

df_trips = df_trips.withColumn(
    "fare_category",
    when(col("fare_amount") < 300, "Low")
    .when(col("fare_amount") < 700, "Medium")
    .otherwise("High")
)

In [0]:
# Extract trip date from trip start timestamp

df_trips = df_trips.withColumn(
    "trip_date",
    to_date(col("trip_start_time"))
)

In [0]:
# Extract year and month for analytical aggregations

df_trips = (
    df_trips
    .withColumn("trip_year", year(col("trip_start_time")))
    .withColumn("trip_month", month(col("trip_start_time")))
)

In [0]:
# Deduplicate records and add processing timestamp

df_trips = trips_obj.dedup(df_trips,["trip_id"],"last_updated_timestamp")
df_trips = trips_obj.process_timestamp(df_trips)


2026-09-10 19:31:06,310 - INFO - Starting deduplication using columns: ['trip_id']
2026-09-10 19:31:07,226 - INFO - Deduplication completed. Input records: 4979, Output records: 4974, Duplicates removed: 5
2026-09-10 19:31:07,227 - INFO - Adding process timestamp to DataFrame
2026-09-10 19:31:07,227 - INFO - Process timestamp added successfully


In [0]:


silver_table = "ride_analytics.silver.trips"

# Check whether Silver table already exists
logger.info("Checking Silver table: %s", silver_table)

# Capture processing start time
start_time = datetime.now()

if not spark.catalog.tableExists(silver_table):

    # First run: create the Silver table
    logger.info("Starting FULL LOAD for Silver Trips")

    df_trips.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(silver_table)

    logger.info(
        "Silver table created successfully: %s",
        silver_table
    )

else:

    # Subsequent runs: perform incremental upsert
    logger.info("Starting INCREMENTAL UPSERT for Silver Trips")

    result = trips_obj.upsert(
        df_trips,
        ["trip_id"],
        silver_table,
        "last_updated_timestamp"
    )

    logger.info(
        "Incremental upsert completed. Merge condition: %s",
        result
    )

# Calculate total processing time
end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

logger.info(
    "Silver Trips processing completed in %.2f seconds",
    duration
)

2026-09-10 19:31:07,675 - INFO - Checking Silver table: ride_analytics.silver.trips
2026-09-10 19:31:07,848 - INFO - Starting FULL LOAD for Silver Trips
2026-09-10 19:31:11,040 - INFO - Silver table created successfully: ride_analytics.silver.trips
2026-09-10 19:31:11,041 - INFO - Silver Trips processing completed in 3.37 seconds


- END